In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error

# Завантаження даних
train_df = pd.read_csv('/kaggle/input/ml-competition-2024-for-ukrainians/train.csv')
test_df = pd.read_csv('/kaggle/input/ml-competition-2024-for-ukrainians/test.csv')


# Заповнення нульових значень у Item_Visibility та логарифмування
train_df["Item_Visibility"] = train_df["Item_Visibility"].replace(0, np.nan)
test_df["Item_Visibility"] = test_df["Item_Visibility"].replace(0, np.nan)
train_df["Item_Visibility"] = np.log1p(train_df["Item_Visibility"].fillna(train_df["Item_Visibility"].mean()))
test_df["Item_Visibility"] = np.log1p(test_df["Item_Visibility"].fillna(train_df["Item_Visibility"].mean()))


# Нормалізація таргету
train_df["Item_Outlet_Sales"] = np.log1p(train_df["Item_Outlet_Sales"])

# Функція для створення Item_MRP_class
def mrp2class(v):
    if v < 70:
        return 0
    elif v < 135:
        return 1
    elif v < 204:
        return 2
    else:
        return 3

train_df["Item_MRP_class"] = train_df["Item_MRP"].apply(mrp2class)
test_df["Item_MRP_class"] = test_df["Item_MRP"].apply(mrp2class)


# Target Encoding
te_cols = ["Item_Identifier", "Item_Type", "Outlet_Identifier"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for col in te_cols:
    train_df[col + "_TE"] = np.nan
    for train_idx, val_idx in kf.split(train_df):
        mean_val = train_df.iloc[train_idx].groupby(col)["Item_Outlet_Sales"].mean()
        train_df.loc[train_df.index[val_idx], col + "_TE"] = train_df.loc[train_df.index[val_idx], col].map(mean_val)
    test_df[col + "_TE"] = test_df[col].map(train_df.groupby(col)["Item_Outlet_Sales"].mean())


# Frequency Encoding
fe_cols = ["Item_Identifier", "Item_Type", "Outlet_Identifier"]
for col in fe_cols:
    freq_map = train_df[col].value_counts().to_dict()
    train_df[col + "_FE"] = train_df[col].map(freq_map)
    test_df[col + "_FE"] = test_df[col].map(freq_map)

# Interaction-based Encoding
def interaction_based_encoding(train_df, test_df,
                               categorical_column,
                               numerical_column,
                               n_splits=5,
                               random_state=42):
    """
    Виконує K-Fold Interaction-Based Encoding для вказаної пари
    (categorical_column, numerical_column). Логіка:
      1) Розбиваємо train на k фолдів
      2) Для кожного фолду рахуємо mean(numerical_column) згідно categorical_column у train-фолді
      3) Застосовуємо ці значення до validation-фолду
      4) Таким чином уникаємо лічка (data leakage)
      5) На test перетворення застосовується за mean-статистикою з усього train
    Повертає оновлені train_df та test_df з додатковою ознакою: <cat>_IE_<num>.
    """
    
    # 1. Ініціюємо KFold
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # 2. Масив для збереження «зашитих» (encoded) значень з кожного фолда
    train_encoded = np.zeros(len(train_df))
    
    # 3. K-Fold цикл
    for train_idx, val_idx in kf.split(train_df):
        # Розбиваємо train_df на train_fold і val_fold
        train_fold = train_df.iloc[train_idx]
        val_fold = train_df.iloc[val_idx]
        
        # 4. Для train_fold рахуємо середнє значення числової колонки
        #    для кожної категорії з categorical_column
        category_stats = train_fold.groupby(categorical_column)[numerical_column].mean()
        
        # 5. Застосовуємо мапінг до val_fold
        #    якщо якась категорія не зустрічалась – .fillna(глобальне середнє)
        val_fold_encoded = val_fold[categorical_column].map(category_stats).fillna(train_df[numerical_column].mean())
        
        # 6. Записуємо отримані значення у загальний масив train_encoded
        train_encoded[val_idx] = val_fold_encoded

    # 7. Додаємо ознаку в train_df
    new_col_name = f"{categorical_column}_IE_{numerical_column}"
    train_df[new_col_name] = train_encoded
    
    # 8. Обчислюємо статистику на **повному** train_df, щоб закодувати test_df
    full_category_stats = train_df.groupby(categorical_column)[numerical_column].mean()
    
    # 9. Застосовуємо до теста
    test_encoded = test_df[categorical_column].map(full_category_stats).fillna(train_df[numerical_column].mean())
    test_df[new_col_name] = test_encoded
    
    # 10. Повертаємо оновлені фрейми
    return train_df, test_df

ie_col_pairs = [
    ("Item_Identifier", "Item_MRP"),
    ("Item_Identifier", "Item_Weight"),
    ("Item_Identifier", "Item_Visibility"),
    ("Item_Type", "Item_MRP"),
    ("Item_Type", "Item_Weight"),
    ("Item_Type", "Item_Visibility"),
    ("Outlet_Identifier", "Item_MRP"),
    ("Outlet_Identifier", "Item_Weight"),
    ("Outlet_Identifier", "Item_Visibility"),
]

# Список, аби зберегти, які колонки ми створимо
ie_cols = []

for cat_c, num_c in ie_col_pairs:
    train_df, test_df = interaction_based_encoding(
        train_df,    # твій train
        test_df,     # твій test
        categorical_column=cat_c,
        numerical_column=num_c,
        n_splits=5,
        random_state=42
    )
    # Нову колонку, яку створила функція, так і називаємо
    new_col_name = f"{cat_c}_IE_{num_c}"
    ie_cols.append(new_col_name)


# Label Encoding
le_cols = ["Item_Identifier", "Item_Fat_Content", "Item_Type", "Outlet_Identifier","Outlet_Size","Outlet_Location_Type","Outlet_Type"]
for col in le_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))


# One-Hot Encoding
ohe_cols = ["Item_Type", "Outlet_Identifier", "Item_MRP_class"]
train_df = pd.get_dummies(train_df, columns=ohe_cols)
test_df = pd.get_dummies(test_df, columns=ohe_cols)


# Standard Scaling
scaler = StandardScaler()
scaled_cols = ["Item_Weight", "Item_Visibility", "Item_MRP"] + [col + "_TE" for col in te_cols]
train_df[scaled_cols] = scaler.fit_transform(train_df[scaled_cols])
test_df[scaled_cols] = scaler.transform(test_df[scaled_cols])


# Підготовка тренувального та тестового наборів
target = train_df["Item_Outlet_Sales"]
train_df.drop(columns=["id", "Item_Outlet_Sales"], inplace=True)
test_df.drop(columns=["id"], inplace=True)
train_df.drop(columns=["Item_Identifier"], inplace=True)
test_df.drop(columns=["Item_Identifier"], inplace=True)

# KFold Cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
test_preds = []

for train_idx, val_idx in kf.split(train_df):
    X_train, X_val = train_df.iloc[train_idx], train_df.iloc[val_idx]
    y_train, y_val = target.iloc[train_idx], target.iloc[val_idx]

    models = {
        "LGBM": LGBMRegressor(n_estimators=1000, learning_rate=0.01),
        "XGB": XGBRegressor(n_estimators=1000, learning_rate=0.01),
        "HGB": HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.01),
        "LR": LinearRegression(),
        "RF": RandomForestRegressor(n_estimators=100, max_depth=10)
    }

    fold_preds = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        fold_preds.append(y_pred)
        print(f"{name} RMSLE:", np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(y_pred))))

    # Ансамблювання через Linear Regression
    ensemble_X = np.column_stack(fold_preds)
    ensemble_model = LinearRegression()
    ensemble_model.fit(ensemble_X, y_val)
    ens_pred = ensemble_model.predict(ensemble_X)
    print("Ensemble RMSLE:", np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(ens_pred))))

    # Передбачення на тестовому наборі
    test_X = np.column_stack([model.predict(test_df) for model in models.values()])
    test_preds.append(ensemble_model.predict(test_X))


# Усереднення передбачень
final_preds = np.mean(test_preds, axis=0)
final_preds = np.expm1(final_preds)  # Повертаємо значення в початковий масштаб

# Збереження у файл
submission = pd.DataFrame({"id": test_df.index, "Item_Outlet_Sales": final_preds})
submission.to_csv("submission.csv", index=False)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013997 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2542
[LightGBM] [Info] Number of data points in the train set: 302742, number of used features: 53
[LightGBM] [Info] Start training from score 7.359671
LGBM RMSLE: 0.7010142137782742
XGB RMSLE: 0.7007488373718606
HGB RMSLE: 0.7010619740532448
LR RMSLE: 0.7089721397703269
RF RMSLE: 0.7017235337476064
Ensemble RMSLE: 0.7003611139813334
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018604 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2538
[LightGBM] [Info] Number of data points in the train set: 302742, number of used features: 53
[LightGBM] [Info] Start training from score 7.360833
LG